# Entraînement CamemBERT v2

Notebook d'entraînement en deux étapes :

1. classification binaire `IA / non-IA` ;
2. classification multi-étiquette des compétences IA uniquement sur les formations IA confirmées.

Le notebook consomme `data/processed/dataset_entrainement.csv` et s'appuie sur le pipeline de nettoyage généré par `scripts/clean_and_merge_datasets.py`.


In [55]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import math

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
    set_seed,
)
from torch import nn


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'processed' / 'dataset_entrainement.csv').exists():
            return candidate
    return start


ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

from scripts.clean_and_merge_datasets import build_text_modele, clean_text, normalize_unicode, parse_multi_values

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('ROOT =', ROOT)
print('DEVICE =', DEVICE)


ROOT = /teamspace/studios/this_studio/deepforma
DEVICE = cpu


## 1. Chargement des données

Le fichier `dataset_entrainement.csv` contient uniquement les exemples IA confirmés et non-IA confirmés.


In [56]:
TRAIN_PATH = ROOT / "data" / "processed" / "ia_multilabel_train.jsonl"
VALIDATION_PATH = ROOT / "data" / "processed" / "ia_multilabel_validation.jsonl"
TEST_PATH = ROOT / "data" / "processed" / "ia_multilabel_test.jsonl"

for path in (TRAIN_PATH, VALIDATION_PATH, TEST_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

train_df = pd.read_json(TRAIN_PATH, lines=True)
validation_df = pd.read_json(VALIDATION_PATH, lines=True)
test_df = pd.read_json(TEST_PATH, lines=True)

print("Train :", train_df.shape)
print("Validation :", validation_df.shape)
print("Test :", test_df.shape)

print("\nColonnes :")
print(train_df.columns.tolist())

display(train_df.head())


Train : (367, 5)
Validation : (63, 5)
Test : (117, 5)

Colonnes :
['text', 'labels', 'multi_hot', 'group_id', 'formation_id']


,text,labels,multi_hot,group_id,formation_id
0,[TITRE] Création de contenus rédactionnels et ...,"[Ethique IA & RGPD, Gestion de projet IA, IA G...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, ...",cert:RS6776,1
1,[TITRE] Intégrer l'intelligence artificielle c...,[IA Generative],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",cert:RS6792,2
2,[TITRE] 100% présentiel - Création de contenus...,"[Deep Learning, Gestion de projet IA, IA Gener...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, ...",cert:RS6776,3
3,[TITRE] CANVA - Créer des supports de communic...,[IA Generative],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...",cert:RS7068,6
4,[TITRE] Intégrer l'Intelligence Artificielle d...,"[Ethique IA & RGPD, Prompt Engineering]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...",cert:RS7288,8


## 2. Préparation du split sans fuite

Les groupes reposent sur `formation_group_id` afin de garder ensemble les doublons exacts et les variantes proches.


In [57]:
def grouped_train_val_test_split(frame: pd.DataFrame, group_col: str = 'formation_group_id', seed: int = 42):
    if group_col not in frame.columns:
        raise KeyError(f'Colonne manquante : {group_col}')

    groups = frame[group_col].astype(str)
    splitter = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed)
    train_idx, temp_idx = next(splitter.split(frame, groups=groups))
    train_df = frame.iloc[train_idx].copy()
    temp_df = frame.iloc[temp_idx].copy()

    temp_groups = temp_df[group_col].astype(str)
    splitter2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed)
    val_idx, test_idx = next(splitter2.split(temp_df, groups=temp_groups))
    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()

    return train_df, val_df, test_df


splits = {
    "train": train_df,
    "val": validation_df,
    "test": test_df,
}
for name, subset in splits.items():
    print(
        f"{name}: "
        f"lignes={len(subset)} "
        f"groupes={subset['group_id'].nunique()}"
    )


assert set(train_df['group_id']).isdisjoint(set(val_df['group_id']))
assert set(train_df['group_id']).isdisjoint(set(test_df['group_id']))
assert set(val_df['group_id']).isdisjoint(set(test_df['group_id']))


train: lignes=367 groupes=205
val: lignes=63 groupes=44
test: lignes=117 groupes=45


## 3. Analyse du déséquilibre

On calcule les distributions de classes pour le modèle binaire et les occurrences de compétences pour le modèle multi-étiquette.


In [58]:
from collections import Counter

# Vérification du schéma réel
print("Colonnes disponibles :", train_df.columns.tolist())
print("Premier enregistrement :", train_df.iloc[0].to_dict())

# Détection de la colonne contenant les labels IA
possible_label_columns = [
    "labels",
    "competences_ia",
    "competences_ia_extraites",
    "target_labels",
]

label_column = next(
    (
        column
        for column in possible_label_columns
        if column in train_df.columns
    ),
    None,
)

if label_column is None:
    raise KeyError(
        "Aucune colonne de labels IA trouvée. "
        f"Colonnes disponibles : {train_df.columns.tolist()}"
    )


def normalize_labels(value):
    if isinstance(value, list):
        return [
            str(label).strip()
            for label in value
            if str(label).strip()
        ]

    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    if isinstance(value, str):
        text = value.strip()

        if not text:
            return []

        try:
            parsed = json.loads(text)

            if isinstance(parsed, list):
                return [
                    str(label).strip()
                    for label in parsed
                    if str(label).strip()
                ]
        except json.JSONDecodeError:
            pass

        return parse_multi_values(text)

    return [str(value).strip()]


train_df["labels_normalized"] = train_df[label_column].apply(
    normalize_labels
)

comp_counter = Counter(
    competence
    for labels in train_df["labels_normalized"]
    for competence in labels
)

comp_stats = pd.DataFrame(
    comp_counter.most_common(),
    columns=["competence", "occurrences"],
)

print("Compétences les plus fréquentes :")
display(comp_stats.head(20))

print("Compétences rares (< 20 occurrences) :")
display(
    comp_stats[
        comp_stats["occurrences"] < 20
    ].sort_values(
        by=["occurrences", "competence"],
        ascending=[True, True],
    )
)

Colonnes disponibles : ['text', 'labels', 'multi_hot', 'group_id', 'formation_id']
Premier enregistrement : {'text': "[TITRE] Création de contenus rédactionnels et visuels par l'usage responsable de l'intelligence artificielle générative - IA en Collectif (17H présentiel + 4H | [SECTEUR] Numérique & Informatique | [ORGANISME] INSTITUT MON ASSISTANT NUMERIQUE | [CERTIFICATION] RS | [NIVEAU] nan | [ROME] nan | [TAGS] Gestion-de-projet-IA | IA-Générative | RS", 'labels': ['Ethique IA & RGPD', 'Gestion de projet IA', 'IA Generative'], 'multi_hot': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'group_id': 'cert:RS6776', 'formation_id': 1}
Compétences les plus fréquentes :


,competence,occurrences
0,IA Generative,161
1,Gestion de projet IA,82
2,Ethique IA & RGPD,74
3,Machine Learning,56
4,Data Science,45
5,Deep Learning,42
6,Big Data,35
7,Prompt Engineering,29
8,Python,29
9,Automatisation,29


Compétences rares (< 20 occurrences) :


,competence,occurrences
19,Reinforcement Learning,13
18,LangChain / Agents RAG,14
17,Series temporelles,14
16,Computer Vision,17
15,NLP / Traitement du langage,17
14,SQL / Data Engineering,17
13,Visualisation,18
12,MLOps / Deploiement,19


## 4. Modèle 1 - classification IA / non-IA

On affine `camembert-base` avec une tête de classification binaire et une perte pondérée pour gérer le déséquilibre.


In [59]:
MODEL_NAME = "camembert-base"
TEXT_COL = "text"
LABEL_COL = "labels_normalized"
MAX_LENGTH = 256

# Utiliser les splits déjà préparés
train_multi = train_df.copy()
val_multi = validation_df.copy()
test_multi = test_df.copy()

# Charger la taxonomie officielle pour garantir l'ordre des sorties
taxonomy_path = ROOT / "config" / "ia_taxonomy_v2.json"

with taxonomy_path.open(encoding="utf-8") as file:
    taxonomy_data = json.load(file)

LABELS = taxonomy_data.get("labels", taxonomy_data)

if not isinstance(LABELS, list):
    raise ValueError(
        "Le fichier de taxonomie doit contenir une liste dans la clé 'labels'."
    )

NUM_LABELS = len(LABELS)

label2id = {
    label: index
    for index, label in enumerate(LABELS)
}

id2label = {
    index: label
    for index, label in enumerate(LABELS)
}

print("Nombre de labels :", NUM_LABELS)
print("Labels :", LABELS)

Nombre de labels : 20
Labels : ['Automatisation', 'Big Data', 'Computer Vision', 'Data Engineering', 'Data Science', 'Deep Learning', 'Ethique IA & RGPD', 'Gestion de projet IA', 'IA Generative', 'LangChain / Agents RAG', 'Machine Learning', 'MLOps / Deploiement', 'NLP / Traitement du langage', 'No-code / Low-code', 'Prompt Engineering', 'Python', 'Reinforcement Learning', 'Series temporelles', 'SQL / Data Engineering', 'Visualisation']


In [60]:
import inspect
import json

import numpy as np
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


# ============================================================
# 1. Configuration
# ============================================================

MODEL_NAME = "camembert-base"
MAX_LENGTH = 256

possible_text_columns = [
    "text",
    "texte_modele",
    "input_text",
    "source_text",
]

possible_label_columns = [
    "labels",
    "labels_normalized",
    "competences_ia",
    "competences_ia_extraites",
    "target_labels",
]

TEXT_COL = next(
    (
        column
        for column in possible_text_columns
        if column in train_df.columns
    ),
    None,
)

LABEL_COL = next(
    (
        column
        for column in possible_label_columns
        if column in train_df.columns
    ),
    None,
)

if TEXT_COL is None:
    raise KeyError(
        "Aucune colonne texte trouvée. "
        f"Colonnes disponibles : {train_df.columns.tolist()}"
    )

if LABEL_COL is None:
    raise KeyError(
        "Aucune colonne de labels trouvée. "
        f"Colonnes disponibles : {train_df.columns.tolist()}"
    )

print("Colonne texte :", TEXT_COL)
print("Colonne labels :", LABEL_COL)


# ============================================================
# 2. Chargement de la taxonomie
# ============================================================

taxonomy_path = ROOT / "config" / "ia_taxonomy_v2.json"

with taxonomy_path.open(encoding="utf-8") as file:
    taxonomy_data = json.load(file)

if isinstance(taxonomy_data, dict):
    LABELS = taxonomy_data.get("labels")
else:
    LABELS = taxonomy_data

if not isinstance(LABELS, list) or not LABELS:
    raise ValueError(
        "La taxonomie doit contenir une liste non vide de labels."
    )

NUM_LABELS = len(LABELS)

label2id = {
    label: index
    for index, label in enumerate(LABELS)
}

id2label = {
    index: label
    for index, label in enumerate(LABELS)
}

print("Nombre de labels :", NUM_LABELS)

if NUM_LABELS != 20:
    print(
        f"Attention : la taxonomie contient {NUM_LABELS} labels, "
        "et non 20."
    )


# ============================================================
# 3. Normalisation des labels
# ============================================================

def normalize_labels(value):
    if isinstance(value, list):
        return [
            str(label).strip()
            for label in value
            if str(label).strip()
        ]

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, str):
        text = value.strip()

        if not text:
            return []

        try:
            parsed = json.loads(text)

            if isinstance(parsed, list):
                return [
                    str(label).strip()
                    for label in parsed
                    if str(label).strip()
                ]
        except json.JSONDecodeError:
            pass

        separators = ["|", ";", ","]

        for separator in separators:
            if separator in text:
                return [
                    item.strip()
                    for item in text.split(separator)
                    if item.strip()
                ]

        return [text]

    return [str(value).strip()]


def encode_multilabel(labels):
    vector = np.zeros(
        NUM_LABELS,
        dtype=np.float32,
    )

    unknown_labels = []

    for label in labels:
        if label not in label2id:
            unknown_labels.append(label)
            continue

        vector[label2id[label]] = 1.0

    if unknown_labels:
        raise ValueError(
            "Labels absents de la taxonomie : "
            f"{sorted(set(unknown_labels))}"
        )

    return vector


# ============================================================
# 4. Préparation des DataFrames
# ============================================================

def prepare_multilabel_dataframe(dataframe, dataset_name):
    if dataframe.empty:
        raise ValueError(
            f"Le DataFrame `{dataset_name}` est vide."
        )

    required_columns = {
        TEXT_COL,
        LABEL_COL,
    }

    missing_columns = (
        required_columns - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            f"Colonnes absentes dans `{dataset_name}` : "
            f"{sorted(missing_columns)}"
        )

    prepared_df = dataframe[
        [TEXT_COL, LABEL_COL]
    ].copy()

    prepared_df[TEXT_COL] = (
        prepared_df[TEXT_COL]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    prepared_df = prepared_df.loc[
        prepared_df[TEXT_COL].ne("")
    ].copy()

    prepared_df["normalized_labels"] = (
        prepared_df[LABEL_COL]
        .apply(normalize_labels)
    )

    prepared_df = prepared_df.loc[
        prepared_df["normalized_labels"].map(bool)
    ].copy()

    prepared_df["labels"] = (
        prepared_df["normalized_labels"]
        .apply(encode_multilabel)
    )

    prepared_df = prepared_df.reset_index(
        drop=True
    )

    print(
        f"{dataset_name}: "
        f"{len(prepared_df)} exemples valides"
    )

    return prepared_df


train_multi = prepare_multilabel_dataframe(
    train_df,
    "train",
)

val_multi = prepare_multilabel_dataframe(
    validation_df,
    "validation",
)

test_multi = prepare_multilabel_dataframe(
    test_df,
    "test",
)


# ============================================================
# 5. Calcul de pos_weight
# ============================================================

y_train = np.stack(
    train_multi["labels"].to_numpy()
).astype(np.float32)

positive_counts = y_train.sum(axis=0)
negative_counts = len(y_train) - positive_counts

pos_weight_values = (
    negative_counts
    / np.maximum(positive_counts, 1.0)
)

pos_weight_values = np.clip(
    pos_weight_values,
    1.0,
    15.0,
)

pos_weight = torch.tensor(
    pos_weight_values,
    dtype=torch.float32,
)

print("\nPoids positifs par label :")

for label, positives, weight in zip(
    LABELS,
    positive_counts,
    pos_weight_values,
):
    print(
        f"{label}: "
        f"positifs={int(positives)}, "
        f"pos_weight={weight:.3f}"
    )


# ============================================================
# 6. Tokenizer et modèle
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    label2id=label2id,
    id2label=id2label,
)

if model.config.num_labels != NUM_LABELS:
    raise ValueError(
        "Nombre de sorties incohérent : "
        f"{model.config.num_labels} au lieu de {NUM_LABELS}"
    )


# ============================================================
# 7. Conversion vers Hugging Face Dataset
# ============================================================

def dataframe_to_dataset(dataframe):
    return Dataset.from_dict(
        {
            "text": dataframe[
                TEXT_COL
            ].tolist(),
            "labels": [
                vector.tolist()
                for vector in dataframe["labels"]
            ],
        }
    )


train_ds = dataframe_to_dataset(train_multi)
val_ds = dataframe_to_dataset(val_multi)
test_ds = dataframe_to_dataset(test_multi)


# ============================================================
# 8. Tokenisation
# ============================================================

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


train_ds = train_ds.map(
    tokenize_batch,
    batched=True,
    desc="Tokenisation train",
)

val_ds = val_ds.map(
    tokenize_batch,
    batched=True,
    desc="Tokenisation validation",
)

test_ds = test_ds.map(
    tokenize_batch,
    batched=True,
    desc="Tokenisation test",
)


def finalize_dataset(dataset):
    removable_columns = [
        column
        for column in dataset.column_names
        if column not in {
            "input_ids",
            "attention_mask",
            "labels",
        }
    ]

    if removable_columns:
        dataset = dataset.remove_columns(
            removable_columns
        )

    dataset.set_format("torch")

    return dataset


train_ds = finalize_dataset(train_ds)
val_ds = finalize_dataset(val_ds)
test_ds = finalize_dataset(test_ds)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)


# ============================================================
# 9. Trainer multilabel pondéré
# ============================================================

class WeightedMultilabelTrainer(Trainer):
    def __init__(
        self,
        *args,
        pos_weight=None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        if pos_weight is None:
            raise ValueError(
                "`pos_weight` est obligatoire."
            )

        self.pos_weight = torch.as_tensor(
            pos_weight,
            dtype=torch.float32,
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).float()

        outputs = model(**model_inputs)
        logits = outputs.logits

        if (
            logits.ndim != 2
            or logits.shape[-1] != NUM_LABELS
        ):
            raise ValueError(
                "Logits attendus sous la forme "
                f"[batch_size, {NUM_LABELS}], "
                f"reçu : {tuple(logits.shape)}"
            )

        loss_function = nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(
                device=logits.device,
                dtype=logits.dtype,
            )
        )

        loss = loss_function(
            logits,
            labels,
        )

        if return_outputs:
            return loss, outputs

        return loss


# ============================================================
# 10. Métriques multilabel
# ============================================================

DEFAULT_THRESHOLD = 0.5


def sigmoid_numpy(logits):
    logits = np.asarray(
        logits,
        dtype=np.float64,
    )

    return 1.0 / (
        1.0 + np.exp(-np.clip(logits, -50, 50))
    )


def compute_multilabel_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    probabilities = sigmoid_numpy(
        predictions
    )

    predicted_labels = (
        probabilities >= DEFAULT_THRESHOLD
    ).astype(np.int32)

    labels = labels.astype(np.int32)

    micro_precision, micro_recall, micro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predicted_labels,
            average="micro",
            zero_division=0,
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        )
    )

    weighted_f1 = f1_score(
        labels,
        predicted_labels,
        average="weighted",
        zero_division=0,
    )

    subset_accuracy = accuracy_score(
        labels,
        predicted_labels,
    )

    empty_prediction_rate = float(
        np.mean(
            predicted_labels.sum(axis=1) == 0
        )
    )

    try:
        average_precision_micro = (
            average_precision_score(
                labels,
                probabilities,
                average="micro",
            )
        )
    except ValueError:
        average_precision_micro = float("nan")

    try:
        average_precision_macro = (
            average_precision_score(
                labels,
                probabilities,
                average="macro",
            )
        )
    except ValueError:
        average_precision_macro = float("nan")

    return {
        "subset_accuracy": subset_accuracy,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "average_precision_micro": average_precision_micro,
        "average_precision_macro": average_precision_macro,
        "empty_prediction_rate": empty_prediction_rate,
    }


# ============================================================
# 11. Arguments d'entraînement
# ============================================================

training_arguments_parameters = inspect.signature(
    TrainingArguments.__init__
).parameters

evaluation_argument = {}

if "eval_strategy" in training_arguments_parameters:
    evaluation_argument["eval_strategy"] = "epoch"
else:
    evaluation_argument["evaluation_strategy"] = "epoch"

use_cuda = torch.cuda.is_available()

use_bf16 = (
    use_cuda
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)

training_args = TrainingArguments(
    output_dir=str(
        ROOT / "models" / "ia-classifier-v2"
    ),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=use_cuda and not use_bf16,
    bf16=use_bf16,
    seed=42,
    data_seed=42,
    max_grad_norm=1.0,
    **evaluation_argument,
)


# ============================================================
# 12. Construction du Trainer
# ============================================================

trainer_parameters = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "data_collator": collator,
    "compute_metrics": compute_multilabel_metrics,
    "pos_weight": pos_weight,
    "callbacks": [
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
}

trainer_signature = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_signature:
    trainer_parameters["processing_class"] = tokenizer
else:
    trainer_parameters["tokenizer"] = tokenizer

trainer = WeightedMultilabelTrainer(
    **trainer_parameters
)

print("Trainer multilabel prêt.")
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de labels :", NUM_LABELS)

Colonne texte : text
Colonne labels : labels
Nombre de labels : 20
train: 367 exemples valides
validation: 63 exemples valides
test: 117 exemples valides

Poids positifs par label :
Automatisation: positifs=29, pos_weight=11.655
Big Data: positifs=35, pos_weight=9.486
Computer Vision: positifs=17, pos_weight=15.000
Data Engineering: positifs=27, pos_weight=12.593
Data Science: positifs=45, pos_weight=7.156
Deep Learning: positifs=42, pos_weight=7.738
Ethique IA & RGPD: positifs=74, pos_weight=3.959
Gestion de projet IA: positifs=82, pos_weight=3.476
IA Generative: positifs=161, pos_weight=1.280
LangChain / Agents RAG: positifs=14, pos_weight=15.000
Machine Learning: positifs=56, pos_weight=5.554
MLOps / Deploiement: positifs=19, pos_weight=15.000
NLP / Traitement du langage: positifs=17, pos_weight=15.000
No-code / Low-code: positifs=23, pos_weight=14.957
Prompt Engineering: positifs=29, pos_weight=11.655
Python: positifs=29, pos_weight=11.655
Reinforcement Learning: positifs=13, pos_w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenisation train:   0%|          | 0/367 [00:00<?, ? examples/s]

Tokenisation validation:   0%|          | 0/63 [00:00<?, ? examples/s]

Tokenisation test:   0%|          | 0/117 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer multilabel prêt.
CUDA disponible : False
Nombre de labels : 20


In [28]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
)


def evaluate_multilabel(
    trainer,
    dataset,
    frame,
    title,
    threshold=0.5,
):
    prediction_output = trainer.predict(dataset)

    logits = prediction_output.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = 1.0 / (
        1.0 + np.exp(
            -np.clip(logits, -50, 50)
        )
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    if "labels" in frame.columns:
        labels = np.stack(
            frame["labels"].to_numpy()
        ).astype(np.int32)

    elif "label_vector" in frame.columns:
        labels = np.stack(
            frame["label_vector"].to_numpy()
        ).astype(np.int32)

    else:
        raise KeyError(
            "Aucune colonne de vecteurs multilabel trouvée. "
            f"Colonnes disponibles : {frame.columns.tolist()}"
        )

    if predictions.shape != labels.shape:
        raise ValueError(
            "Dimensions incompatibles entre prédictions et labels : "
            f"{predictions.shape} contre {labels.shape}"
        )

    micro_precision, micro_recall, micro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="micro",
            zero_division=0,
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        )
    )

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    subset_accuracy = accuracy_score(
        labels,
        predictions,
    )

    empty_prediction_rate = float(
        np.mean(
            predictions.sum(axis=1) == 0
        )
    )

    average_labels_expected = float(
        labels.sum(axis=1).mean()
    )

    average_labels_predicted = float(
        predictions.sum(axis=1).mean()
    )

    confusion_matrices = (
        multilabel_confusion_matrix(
            labels,
            predictions,
        )
    )

    print(f"\n{title}")
    print("=" * len(title))
    print(f"Seuil : {threshold:.2f}")
    print(f"Subset accuracy : {subset_accuracy:.4f}")
    print(f"Micro precision : {micro_precision:.4f}")
    print(f"Micro recall : {micro_recall:.4f}")
    print(f"Micro F1 : {micro_f1:.4f}")
    print(f"Macro precision : {macro_precision:.4f}")
    print(f"Macro recall : {macro_recall:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")
    print(f"Weighted F1 : {weighted_f1:.4f}")
    print(
        "Taux de prédictions vides : "
        f"{empty_prediction_rate:.4f}"
    )
    print(
        "Nombre moyen de labels attendus : "
        f"{average_labels_expected:.2f}"
    )
    print(
        "Nombre moyen de labels prédits : "
        f"{average_labels_predicted:.2f}"
    )

    report = classification_report(
        labels,
        predictions,
        target_names=LABELS,
        zero_division=0,
        output_dict=True,
    )

    per_label_rows = []

    for index, label in enumerate(LABELS):
        matrix = confusion_matrices[index]

        tn, fp, fn, tp = matrix.ravel()

        label_metrics = report[label]

        per_label_rows.append(
            {
                "label": label,
                "precision": label_metrics["precision"],
                "recall": label_metrics["recall"],
                "f1": label_metrics["f1-score"],
                "support": int(label_metrics["support"]),
                "true_positive": int(tp),
                "false_positive": int(fp),
                "false_negative": int(fn),
                "true_negative": int(tn),
            }
        )

    per_label_df = pd.DataFrame(
        per_label_rows
    ).sort_values(
        by="f1",
        ascending=False,
    )

    display(per_label_df)

    return {
        "predictions": predictions,
        "probabilities": probabilities,
        "labels": labels,
        "per_label": per_label_df,
        "confusion_matrices": confusion_matrices,
        "metrics": {
            "subset_accuracy": subset_accuracy,
            "micro_precision": micro_precision,
            "micro_recall": micro_recall,
            "micro_f1": micro_f1,
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "empty_prediction_rate": empty_prediction_rate,
        },
    }

## 5. Modèle 2 - multi-étiquette des compétences IA

Le second modèle est entraîné uniquement sur les formations IA confirmées. Les cibles sont binarisées avec `MultiLabelBinarizer`.


In [32]:
import json

import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


# ============================================================
# 1. Utiliser directement les splits préparés
# ============================================================

ia_train = train_df.copy()
ia_val = validation_df.copy()
ia_test = test_df.copy()

if ia_train.empty:
    raise ValueError("Le split d'entraînement est vide.")

if ia_val.empty:
    raise ValueError("Le split de validation est vide.")

if ia_test.empty:
    raise ValueError("Le split de test est vide.")


# ============================================================
# 2. Détecter la colonne de labels
# ============================================================

possible_label_columns = [
    "labels",
    "labels_normalized",
    "competences_ia",
    "competences_ia_extraites",
    "target_labels",
]

LABEL_COL = next(
    (
        column
        for column in possible_label_columns
        if column in ia_train.columns
    ),
    None,
)

if LABEL_COL is None:
    raise KeyError(
        "Aucune colonne de labels trouvée. "
        f"Colonnes disponibles : {ia_train.columns.tolist()}"
    )

print("Colonne de labels utilisée :", LABEL_COL)


# ============================================================
# 3. Normaliser les labels
# ============================================================

def normalize_multilabel_value(value):
    if isinstance(value, list):
        return [
            str(label).strip()
            for label in value
            if str(label).strip()
        ]

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, str):
        text = value.strip()

        if not text:
            return []

        try:
            parsed = json.loads(text)

            if isinstance(parsed, list):
                return [
                    str(label).strip()
                    for label in parsed
                    if str(label).strip()
                ]
        except json.JSONDecodeError:
            pass

        for separator in ("|", ";", ","):
            if separator in text:
                return [
                    item.strip()
                    for item in text.split(separator)
                    if item.strip()
                ]

        return [text]

    return [str(value).strip()]


train_labels = [
    normalize_multilabel_value(value)
    for value in ia_train[LABEL_COL]
]

val_labels = [
    normalize_multilabel_value(value)
    for value in ia_val[LABEL_COL]
]

test_labels = [
    normalize_multilabel_value(value)
    for value in ia_test[LABEL_COL]
]


# ============================================================
# 4. Charger la taxonomie officielle
# ============================================================

taxonomy_path = ROOT / "config" / "ia_taxonomy_v2.json"

with taxonomy_path.open(encoding="utf-8") as file:
    taxonomy_data = json.load(file)

if isinstance(taxonomy_data, dict):
    LABELS = taxonomy_data.get("labels")
else:
    LABELS = taxonomy_data

if not isinstance(LABELS, list) or not LABELS:
    raise ValueError(
        "La taxonomie IA doit contenir une liste non vide de labels."
    )

print("Nombre de compétences :", len(LABELS))
print("Compétences :", LABELS)


# ============================================================
# 5. Encodage multilabel avec ordre fixe
# ============================================================

mlb = MultiLabelBinarizer(
    classes=LABELS
)

# Nécessaire pour initialiser classes_
mlb.fit([LABELS])

Y_train = mlb.transform(train_labels)
Y_val = mlb.transform(val_labels)
Y_test = mlb.transform(test_labels)

print("Shape train :", Y_train.shape)
print("Shape validation :", Y_val.shape)
print("Shape test :", Y_test.shape)


# ============================================================
# 6. Vérifier les labels inconnus
# ============================================================

known_labels = set(LABELS)

unknown_labels = sorted(
    {
        label
        for split_labels in (
            train_labels,
            val_labels,
            test_labels,
        )
        for labels in split_labels
        for label in labels
        if label not in known_labels
    }
)

if unknown_labels:
    raise ValueError(
        "Certains labels du dataset sont absents de la taxonomie : "
        f"{unknown_labels}"
    )


# ============================================================
# 7. Tokenizer et modèle
# ============================================================

label2id = {
    label: index
    for index, label in enumerate(LABELS)
}

id2label = {
    index: label
    for index, label in enumerate(LABELS)
}

tokenizer_ml = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model_ml = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type="multi_label_classification",
    label2id=label2id,
    id2label=id2label,
)

print("Nombre de sorties du modèle :", model_ml.config.num_labels)
print("Type de problème :", model_ml.config.problem_type)

Colonne de labels utilisée : labels
Nombre de compétences : 20
Compétences : ['Automatisation', 'Big Data', 'Computer Vision', 'Data Engineering', 'Data Science', 'Deep Learning', 'Ethique IA & RGPD', 'Gestion de projet IA', 'IA Generative', 'LangChain / Agents RAG', 'Machine Learning', 'MLOps / Deploiement', 'NLP / Traitement du langage', 'No-code / Low-code', 'Prompt Engineering', 'Python', 'Reinforcement Learning', 'Series temporelles', 'SQL / Data Engineering', 'Visualisation']
Shape train : (367, 20)
Shape validation : (63, 20)
Shape test : (117, 20)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Nombre de sorties du modèle : 20
Type de problème : multi_label_classification


In [33]:
import inspect

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


# ============================================================
# 1. Vérifications
# ============================================================

NUM_LABELS = Y_train.shape[1]

if len(ia_train) != len(Y_train):
    raise ValueError(
        f"Décalage train : {len(ia_train)} textes "
        f"contre {len(Y_train)} vecteurs de labels."
    )

if len(ia_val) != len(Y_val):
    raise ValueError(
        f"Décalage validation : {len(ia_val)} textes "
        f"contre {len(Y_val)} vecteurs de labels."
    )

if len(ia_test) != len(Y_test):
    raise ValueError(
        f"Décalage test : {len(ia_test)} textes "
        f"contre {len(Y_test)} vecteurs de labels."
    )

if Y_train.ndim != 2:
    raise ValueError(
        f"Y_train doit être une matrice 2D. "
        f"Forme reçue : {Y_train.shape}"
    )

if getattr(model_ml.config, "num_labels", None) != NUM_LABELS:
    raise ValueError(
        "Le nombre de sorties du modèle ne correspond pas "
        "au nombre de compétences.\n"
        f"model_ml.config.num_labels = {model_ml.config.num_labels}\n"
        f"Y_train.shape[1] = {NUM_LABELS}"
    )

print(f"Nombre de compétences : {NUM_LABELS}")
print(f"Train IA : {len(ia_train)}")
print(f"Validation IA : {len(ia_val)}")
print(f"Test IA : {len(ia_test)}")


# ============================================================
# 2. Construction des DataFrames
# ============================================================

def create_multilabel_dataframe(frame, labels):
    """
    Crée un DataFrame texte + vecteur multi-étiquette.
    """

    if frame is None or len(frame) == 0:
        return None

    if len(frame) != len(labels):
        raise ValueError(
            f"Nombre de textes différent du nombre de labels : "
            f"{len(frame)} contre {len(labels)}."
        )

    texts = (
        frame[TEXT_COL]
        .fillna("")
        .astype(str)
        .str.strip()
        .tolist()
    )

    labels_array = np.asarray(
        labels,
        dtype=np.float32,
    )

    valid_indices = [
        index
        for index, text in enumerate(texts)
        if text
    ]

    if not valid_indices:
        return None

    clean_texts = [
        texts[index]
        for index in valid_indices
    ]

    clean_labels = labels_array[
        valid_indices
    ]

    return pd.DataFrame(
        {
            TEXT_COL: clean_texts,
            "labels": clean_labels.tolist(),
        }
    )


train_ml_df = create_multilabel_dataframe(
    ia_train,
    Y_train,
)

val_ml_df = create_multilabel_dataframe(
    ia_val,
    Y_val,
)

test_ml_df = create_multilabel_dataframe(
    ia_test,
    Y_test,
)

if train_ml_df is None or train_ml_df.empty:
    raise ValueError(
        "Le jeu d'entraînement multi-étiquette est vide."
    )


# ============================================================
# 3. Conversion en Dataset Hugging Face
# ============================================================

train_ml = Dataset.from_pandas(
    train_ml_df,
    preserve_index=False,
)

val_ml = (
    Dataset.from_pandas(
        val_ml_df,
        preserve_index=False,
    )
    if val_ml_df is not None
    else None
)

test_ml = (
    Dataset.from_pandas(
        test_ml_df,
        preserve_index=False,
    )
    if test_ml_df is not None
    else None
)


# ============================================================
# 4. Tokenisation
# ============================================================

def tokenize_multilabel(batch):
    return tokenizer_ml(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


train_ml = train_ml.map(
    tokenize_multilabel,
    batched=True,
    desc="Tokenisation multi-label train",
)

if val_ml is not None:
    val_ml = val_ml.map(
        tokenize_multilabel,
        batched=True,
        desc="Tokenisation multi-label validation",
    )

if test_ml is not None:
    test_ml = test_ml.map(
        tokenize_multilabel,
        batched=True,
        desc="Tokenisation multi-label test",
    )


def remove_text_column(dataset):
    if dataset is None:
        return None

    if TEXT_COL in dataset.column_names:
        dataset = dataset.remove_columns(
            [TEXT_COL]
        )

    return dataset


train_ml = remove_text_column(train_ml)
val_ml = remove_text_column(val_ml)
test_ml = remove_text_column(test_ml)


# ============================================================
# 5. Calcul des poids positifs
# ============================================================

train_label_counts = np.asarray(
    Y_train,
    dtype=np.float32,
).sum(axis=0)

negative_counts = (
    len(Y_train) - train_label_counts
)

raw_pos_weight = (
    negative_counts
    / np.clip(
        train_label_counts,
        1.0,
        None,
    )
)

# Évite qu'une compétence très rare domine entièrement la loss.
MAX_POS_WEIGHT = 20.0

clipped_pos_weight = np.clip(
    raw_pos_weight,
    1.0,
    MAX_POS_WEIGHT,
)

pos_weight = torch.tensor(
    clipped_pos_weight,
    dtype=torch.float32,
)

print("\nOccurrences par compétence :")
print(train_label_counts.astype(int).tolist())

print("\nPoids positifs utilisés :")
print(pos_weight.tolist())


# ============================================================
# 6. Trainer pondéré multi-étiquette
# ============================================================

class WeightedMultiLabelTrainer(Trainer):
    def __init__(self, pos_weight=None, **kwargs):
        super().__init__(**kwargs)

        if pos_weight is None:
            pos_weight = torch.ones(
                NUM_LABELS,
                dtype=torch.float32,
            )

        self.pos_weight = torch.as_tensor(
            pos_weight,
            dtype=torch.float32,
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        # Ne pas modifier directement le dictionnaire reçu.
        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).float()

        outputs = model(**model_inputs)
        logits = outputs.logits

        if logits.ndim != 2:
            raise ValueError(
                "Les logits doivent être une matrice 2D "
                f"[batch_size, num_labels]. Reçu : {logits.shape}"
            )

        if logits.shape[-1] != NUM_LABELS:
            raise ValueError(
                f"Le modèle produit {logits.shape[-1]} sorties, "
                f"mais {NUM_LABELS} compétences sont attendues."
            )

        weights = self.pos_weight.to(
            device=logits.device,
            dtype=logits.dtype,
        )

        loss_function = nn.BCEWithLogitsLoss(
            pos_weight=weights,
        )

        loss = loss_function(
            logits,
            labels,
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


# ============================================================
# 7. Métriques multi-étiquette
# ============================================================

MULTILABEL_THRESHOLD = 0.5


def sigmoid_numpy(values):
    """
    Sigmoïde numériquement stable.
    """
    values = np.clip(
        values,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-values)
    )


def compute_multilabel_metrics(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(logits, tuple):
        logits = logits[0]

    labels = np.asarray(
        labels,
        dtype=np.int32,
    )

    probabilities = sigmoid_numpy(logits)

    preds = (
        probabilities >= MULTILABEL_THRESHOLD
    ).astype(np.int32)

    precision_micro, recall_micro, f1_micro, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="micro",
            zero_division=0,
        )
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="macro",
            zero_division=0,
        )
    )

    precision_samples, recall_samples, f1_samples, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="samples",
            zero_division=0,
        )
    )

    exact_match = np.mean(
        np.all(
            labels == preds,
            axis=1,
        )
    )

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_samples": f1_samples,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "precision_samples": precision_samples,
        "recall_samples": recall_samples,
        "exact_match": exact_match,
    }


# ============================================================
# 8. Compatibilité Transformers
# ============================================================

training_arguments_parameters = inspect.signature(
    TrainingArguments.__init__
).parameters

evaluation_argument = {}

if "eval_strategy" in training_arguments_parameters:
    evaluation_argument["eval_strategy"] = "epoch"
else:
    evaluation_argument["evaluation_strategy"] = "epoch"


use_cuda = torch.cuda.is_available()

use_bf16 = (
    use_cuda
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)


# Early stopping et load_best_model nécessitent un jeu validation.
has_validation = (
    val_ml is not None
    and len(val_ml) > 0
)

training_arguments_values = {
    "output_dir": str(
        ROOT
        / "models"
        / "multilabel_competences_v2"
    ),
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "num_train_epochs": 4,
    "weight_decay": 0.01,
    "logging_strategy": "steps",
    "logging_steps": 20,
    "save_total_limit": 2,
    "report_to": "none",
    "fp16": use_cuda and not use_bf16,
    "bf16": use_bf16,
    "seed": 42,
    "data_seed": 42,
}

if has_validation:
    training_arguments_values.update(
        {
            "save_strategy": "epoch",
            "load_best_model_at_end": True,
            "metric_for_best_model": "f1_micro",
            "greater_is_better": True,
            **evaluation_argument,
        }
    )
else:
    training_arguments_values.update(
        {
            "save_strategy": "no",
            "load_best_model_at_end": False,
        }
    )

multi_args = TrainingArguments(
    **training_arguments_values
)


# ============================================================
# 9. Construction du Trainer
# ============================================================

trainer_parameters = {
    "model": model_ml,
    "args": multi_args,
    "train_dataset": train_ml,
    "eval_dataset": val_ml,
    "data_collator": DataCollatorWithPadding(
        tokenizer=tokenizer_ml,
        return_tensors="pt",
    ),
    "compute_metrics": (
        compute_multilabel_metrics
        if has_validation
        else None
    ),
    "pos_weight": pos_weight,
}

if has_validation:
    trainer_parameters["callbacks"] = [
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]

trainer_signature = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_signature:
    trainer_parameters[
        "processing_class"
    ] = tokenizer_ml
else:
    trainer_parameters[
        "tokenizer"
    ] = tokenizer_ml


multi_trainer = WeightedMultiLabelTrainer(
    **trainer_parameters
)

print("\nTrainer multi-étiquette prêt.")

Nombre de compétences : 20
Train IA : 367
Validation IA : 63
Test IA : 117


Tokenisation multi-label train:   0%|          | 0/367 [00:00<?, ? examples/s]

Tokenisation multi-label validation:   0%|          | 0/63 [00:00<?, ? examples/s]

Tokenisation multi-label test:   0%|          | 0/117 [00:00<?, ? examples/s]


Occurrences par compétence :
[29, 35, 17, 27, 45, 42, 74, 82, 161, 14, 56, 19, 17, 23, 29, 29, 13, 14, 17, 18]

Poids positifs utilisés :
[11.655172348022461, 9.485713958740234, 20.0, 12.592592239379883, 7.155555725097656, 7.738095283508301, 3.9594595432281494, 3.47560977935791, 1.2795031070709229, 20.0, 5.5535712242126465, 18.3157901763916, 20.0, 14.956521987915039, 11.655172348022461, 11.655172348022461, 20.0, 20.0, 20.0, 19.38888931274414]

Trainer multi-étiquette prêt.


In [50]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)


def stable_sigmoid(logits):
    """
    Calcule une sigmoïde stable numériquement.
    """
    logits = np.clip(logits, -50, 50)
    return 1.0 / (1.0 + np.exp(-logits))


def evaluate_multilabel(
    trainer,
    dataset,
    mlb,
    threshold=0.5,
    title="Évaluation multi-étiquette",
):
    """
    Évalue un modèle multi-étiquette à partir des labels stockés
    directement dans le Dataset Hugging Face.

    Cette fonction ne dépend pas d'une colonne `competences_ia`
    dans un DataFrame Pandas.
    """

    if dataset is None:
        raise ValueError("Le dataset multi-étiquette est absent.")

    if len(dataset) == 0:
        raise ValueError("Le dataset multi-étiquette est vide.")

    if "labels" not in dataset.column_names:
        raise KeyError(
            "Le Dataset Hugging Face ne contient pas de colonne `labels`."
        )

    prediction_output = trainer.predict(dataset)

    logits = prediction_output.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = stable_sigmoid(logits)

    predictions = (
        probabilities >= threshold
    ).astype(np.int32)

    # Utiliser en priorité les labels renvoyés par Trainer
    true_labels = prediction_output.label_ids

    if true_labels is None:
        true_labels = np.asarray(
            dataset["labels"],
            dtype=np.int32,
        )
    else:
        true_labels = np.asarray(
            true_labels,
            dtype=np.int32,
        )

    if predictions.shape != true_labels.shape:
        raise ValueError(
            "La forme des prédictions ne correspond pas à celle des labels : "
            f"{predictions.shape} contre {true_labels.shape}."
        )

    if predictions.shape[1] != len(mlb.classes_):
        raise ValueError(
            "Le nombre de sorties du modèle ne correspond pas au nombre "
            "de classes du MultiLabelBinarizer : "
            f"{predictions.shape[1]} sorties contre "
            f"{len(mlb.classes_)} classes."
        )

    precision_micro, recall_micro, f1_micro, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="micro",
            zero_division=0,
        )
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="macro",
            zero_division=0,
        )
    )

    precision_samples, recall_samples, f1_samples, _ = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average="samples",
            zero_division=0,
        )
    )

    exact_match = np.mean(
        np.all(
            true_labels == predictions,
            axis=1,
        )
    )

    predicted_count_per_sample = predictions.sum(axis=1)
    true_count_per_sample = true_labels.sum(axis=1)

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print(f"Seuil                 : {threshold:.3f}")
    print(f"Nombre d'exemples     : {len(true_labels)}")
    print(f"Nombre de compétences : {len(mlb.classes_)}")

    print("\nMétriques globales :")
    print(f"Précision micro : {precision_micro:.4f}")
    print(f"Rappel micro    : {recall_micro:.4f}")
    print(f"F1 micro        : {f1_micro:.4f}")

    print(f"\nPrécision macro : {precision_macro:.4f}")
    print(f"Rappel macro    : {recall_macro:.4f}")
    print(f"F1 macro        : {f1_macro:.4f}")

    print(f"\nF1 samples      : {f1_samples:.4f}")
    print(f"Exact match     : {exact_match:.4f}")

    print(
        "\nNombre moyen de compétences réelles par formation : "
        f"{true_count_per_sample.mean():.2f}"
    )

    print(
        "Nombre moyen de compétences prédites par formation : "
        f"{predicted_count_per_sample.mean():.2f}"
    )

    # Statistiques classe par classe
    class_precision, class_recall, class_f1, class_support = (
        precision_recall_fscore_support(
            true_labels,
            predictions,
            average=None,
            zero_division=0,
        )
    )

    per_class_metrics = pd.DataFrame(
        {
            "competence": mlb.classes_,
            "support": class_support.astype(int),
            "precision": class_precision,
            "recall": class_recall,
            "f1": class_f1,
            "predictions_positives": predictions.sum(axis=0),
        }
    )

    per_class_metrics = per_class_metrics.sort_values(
        by=["f1", "support"],
        ascending=[True, True],
    ).reset_index(drop=True)

    print("\nCompétences les plus difficiles :")
    display(per_class_metrics.head(15))

    print("\nRapport de classification :")
    print(
        classification_report(
            true_labels,
            predictions,
            target_names=[
                str(label)
                for label in mlb.classes_
            ],
            zero_division=0,
        )
    )

    true_competences = mlb.inverse_transform(
        true_labels
    )

    predicted_competences = mlb.inverse_transform(
        predictions
    )

    detailed_predictions = pd.DataFrame(
        {
            "competences_reelles": [
                list(values)
                for values in true_competences
            ],
            "competences_predites": [
                list(values)
                for values in predicted_competences
            ],
            "nombre_reel": true_count_per_sample,
            "nombre_predit": predicted_count_per_sample,
        }
    )

    metrics = {
        "threshold": float(threshold),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_micro": float(f1_micro),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_samples": float(precision_samples),
        "recall_samples": float(recall_samples),
        "f1_samples": float(f1_samples),
        "exact_match": float(exact_match),
        "mean_true_labels": float(
            true_count_per_sample.mean()
        ),
        "mean_predicted_labels": float(
            predicted_count_per_sample.mean()
        ),
    }

    return {
        "metrics": metrics,
        "predictions": predictions,
        "probabilities": probabilities,
        "true_labels": true_labels,
        "per_class_metrics": per_class_metrics,
        "detailed_predictions": detailed_predictions,
    }

## 6. Fonction de prédiction

La prédiction s'effectue en cascade :

- le modèle binaire calcule `probabilite_ia` ;
- si la probabilité est sous le seuil, le modèle de compétences n'est pas sollicité ;
- sinon le second modèle renvoie les compétences au-dessus du seuil.


In [49]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch

DEFAULT_COMPETENCE_THRESHOLD = 0.50
MIN_THRESHOLD = 0.15
MAX_THRESHOLD = 0.85


def build_model_input(formation_data):
    if isinstance(formation_data, pd.Series):
        row = formation_data.copy()
    elif isinstance(formation_data, dict):
        row = pd.Series(formation_data)
    else:
        raise TypeError(
            "formation_data doit être un dictionnaire ou une Series pandas."
        )

    existing_text = clean_text(
        row.get("text", row.get("texte_modele", ""))
    )

    if existing_text:
        row["text"] = existing_text
        return row

    base = {
        "intitule": row.get(
            "intitule",
            row.get("Intitulé", row.get("Intitulé de la formation", "")),
        ),
        "description": row.get("description", ""),
        "objectifs": row.get("objectifs", ""),
        "programme": row.get("programme", ""),
        "public_cible": row.get(
            "public_cible",
            row.get("Public cible", ""),
        ),
        "prerequis": row.get("prerequis", ""),
        "niveau": row.get("niveau", row.get("Niveau", "")),
        "modalite": row.get("modalite", row.get("Modalité", "")),
        "duree": row.get("duree", row.get("Durée", "")),
        "certification": row.get(
            "certification",
            row.get("Type de certification", ""),
        ),
        "codes_rome": row.get("codes_rome", row.get("Codes ROME", "")),
        "organisme": row.get(
            "organisme",
            row.get("Organisme de formation", ""),
        ),
        "secteur": row.get("secteur", row.get("Secteur", "")),
        "tags": row.get("tags", row.get("Tags TrendRadar", "")),
    }

    model_row = pd.Series(base)
    generated_text = clean_text(build_text_modele(model_row))

    if not generated_text:
        raise ValueError(
            "Impossible de construire un texte exploitable "
            "à partir des données de la formation."
        )

    model_row["text"] = generated_text
    return model_row


def load_thresholds(
    thresholds_path,
    labels,
    default_threshold=DEFAULT_COMPETENCE_THRESHOLD,
):
    thresholds = {
        label: float(default_threshold)
        for label in labels
    }

    if thresholds_path is None:
        return thresholds

    path = Path(thresholds_path)

    if not path.exists():
        print(
            f"Avertissement : seuils introuvables dans {path}. "
            f"Seuil par défaut utilisé : {default_threshold:.2f}"
        )
        return thresholds

    with path.open("r", encoding="utf-8") as file:
        loaded = json.load(file)

    if isinstance(loaded, dict) and "thresholds" in loaded:
        loaded = loaded["thresholds"]

    if not isinstance(loaded, dict):
        raise ValueError(
            "Le fichier de seuils doit contenir un dictionnaire."
        )

    for label in labels:
        value = loaded.get(label, default_threshold)

        try:
            value = float(value)
        except (TypeError, ValueError):
            value = default_threshold

        thresholds[label] = float(
            np.clip(value, MIN_THRESHOLD, MAX_THRESHOLD)
        )

    return thresholds


def predict_formation(
    formation_data,
    model,
    tokenizer,
    labels,
    device,
    thresholds=None,
    default_threshold=DEFAULT_COMPETENCE_THRESHOLD,
    max_length=256,
    top_k=None,
):
    row = build_model_input(formation_data)
    text = clean_text(row.get("text", row.get("texte_modele", "")))

    if not text:
        raise ValueError("Le texte d'entrée est vide.")

    if thresholds is None:
        thresholds = {
            label: default_threshold
            for label in labels
        }

    model = model.to(device)
    model.eval()

    encoded_inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True,
    )

    encoded_inputs = {
        key: value.to(device)
        for key, value in encoded_inputs.items()
    }

    with torch.no_grad():
        outputs = model(**encoded_inputs)
        probabilities = (
            torch.sigmoid(outputs.logits)
            .detach()
            .cpu()
            .numpy()[0]
        )

    if len(probabilities) != len(labels):
        raise ValueError(
            f"{len(probabilities)} probabilités pour {len(labels)} labels."
        )

    all_predictions = []

    for label, probability in zip(labels, probabilities):
        threshold = float(
            thresholds.get(label, default_threshold)
        )
        threshold = float(
            np.clip(threshold, MIN_THRESHOLD, MAX_THRESHOLD)
        )

        all_predictions.append(
            {
                "nom": label,
                "probabilite": float(probability),
                "seuil": threshold,
                "selectionnee": bool(probability >= threshold),
            }
        )

    all_predictions.sort(
        key=lambda item: item["probabilite"],
        reverse=True,
    )

    selected_predictions = [
        prediction
        for prediction in all_predictions
        if prediction["selectionnee"]
    ]

    if top_k is not None:
        if top_k <= 0:
            raise ValueError("top_k doit être strictement positif.")
        selected_predictions = selected_predictions[:top_k]

    max_probability = (
        float(all_predictions[0]["probabilite"])
        if all_predictions
        else 0.0
    )

    return {
        "est_lie_ia": bool(selected_predictions),
        "probabilite_ia_max": max_probability,
        "nombre_competences_detectees": len(selected_predictions),
        "competences": selected_predictions,
        "scores_complets": all_predictions,
        "texte_modele": text,
    }


thresholds_path = (
    ROOT
    / "models"
    / "ia-classifier-v2"
    / "final"
    / "thresholds.json"
)

thresholds = load_thresholds(
    thresholds_path=thresholds_path,
    labels=LABELS,
    default_threshold=DEFAULT_COMPETENCE_THRESHOLD,
)

example_result = predict_formation(
    formation_data={
        "intitule": "Formation IA générative pour les ressources humaines",
        "description": (
            "Utiliser les modèles de langage, rédiger des prompts "
            "et automatiser certaines tâches RH."
        ),
        "public_cible": (
            "Responsables et équipes des ressources humaines"
        ),
    },
    model=model_ml,
    tokenizer=tokenizer_ml,
    labels=LABELS,
    device=DEVICE,
    thresholds=thresholds,
    max_length=MAX_LENGTH,
    top_k=5,
)

example_result


Avertissement : seuils introuvables dans /teamspace/studios/this_studio/deepforma/models/ia-classifier-v2/final/thresholds.json. Seuil par défaut utilisé : 0.50


{'est_lie_ia': True,
 'probabilite_ia_max': 0.5196933150291443,
 'nombre_competences_detectees': 5,
 'competences': [{'nom': 'Data Science',
   'probabilite': 0.5196933150291443,
   'seuil': 0.5,
   'selectionnee': True},
  {'nom': 'LangChain / Agents RAG',
   'probabilite': 0.5196627378463745,
   'seuil': 0.5,
   'selectionnee': True},
  {'nom': 'No-code / Low-code',
   'probabilite': 0.5172815322875977,
   'seuil': 0.5,
   'selectionnee': True},
  {'nom': 'Computer Vision',
   'probabilite': 0.5155252814292908,
   'seuil': 0.5,
   'selectionnee': True},
  {'nom': 'Ethique IA & RGPD',
   'probabilite': 0.5141386985778809,
   'seuil': 0.5,
   'selectionnee': True}],
 'scores_complets': [{'nom': 'Data Science',
   'probabilite': 0.5196933150291443,
   'seuil': 0.5,
   'selectionnee': True},
  {'nom': 'LangChain / Agents RAG',
   'probabilite': 0.5196627378463745,
   'seuil': 0.5,
   'selectionnee': True},
  {'nom': 'No-code / Low-code',
   'probabilite': 0.5172815322875977,
   'seuil': 

## 7. Évaluation finale et sauvegarde

Une fois l'entraînement terminé, évaluez sur le test split, sauvegardez les modèles et documentez les seuils retenus.


In [54]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# ============================================================
# 1. Vérifications
# ============================================================

required_variables = {
    "trainer": trainer,
    "test_ds": test_ds,
    "test_multi": test_multi,
    "tokenizer": tokenizer,
    "LABELS": LABELS,
    "ROOT": ROOT,
}

for variable_name, variable_value in required_variables.items():
    if variable_value is None:
        raise ValueError(
            f"La variable `{variable_name}` est absente ou vaut None."
        )

if len(test_ds) == 0:
    raise ValueError("Le jeu de test multilabel est vide.")

print("Variables de test disponibles.")


# ============================================================
# 2. Évaluation du modèle multilabel
# ============================================================

print("\n" + "=" * 60)
print("ÉVALUATION DU MODÈLE MULTILABEL")
print("=" * 60)

test_metrics = trainer.evaluate(
    eval_dataset=test_ds,
    metric_key_prefix="test",
)

print("\nMétriques retournées par Trainer :")

for metric_name, metric_value in test_metrics.items():
    if isinstance(
        metric_value,
        (int, float, np.integer, np.floating),
    ):
        print(
            f"{metric_name}: "
            f"{float(metric_value):.4f}"
        )
    else:
        print(
            f"{metric_name}: {metric_value}"
        )


# ============================================================
# 3. Évaluation détaillée
# ============================================================

test_results = evaluate_multilabel(
    trainer=trainer,
    dataset=test_ds,
    mlb=mlb,
    threshold=0.50,
    title="Test multilabel",
)


# ============================================================
# 4. Dossier final
# ============================================================

final_dir = (
    ROOT
    / "models"
    / "ia-classifier-v2"
    / "final"
)

final_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5. Sauvegarde du modèle
# ============================================================

trainer.save_model(
    str(final_dir)
)

tokenizer.save_pretrained(
    str(final_dir)
)


# ============================================================
# 6. Sauvegarde de la taxonomie
# ============================================================

taxonomy_output = {
    "version": "2.0",
    "labels": LABELS,
}

with (
    final_dir / "taxonomy.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        taxonomy_output,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 7. Sauvegarde des seuils
# ============================================================

threshold_config = {
    "default_threshold": 0.50,
    "thresholds": {
        label: 0.50
        for label in LABELS
    },
}

with (
    final_dir / "thresholds.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        threshold_config,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 8. Conversion JSON des métriques
# ============================================================

def make_json_serializable(value):
    if value is None:
        return None

    if isinstance(
        value,
        (str, bool, int, float),
    ):
        return value

    if isinstance(value, dict):
        return {
            str(key): make_json_serializable(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            make_json_serializable(item)
            for item in value
        ]

    if isinstance(value, pd.DataFrame):
        return make_json_serializable(
            value.to_dict(
                orient="records"
            )
        )

    if isinstance(value, pd.Series):
        return make_json_serializable(
            value.to_list()
        )

    if isinstance(value, np.ndarray):
        return make_json_serializable(
            value.tolist()
        )

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        numeric_value = float(value)

        if not np.isfinite(numeric_value):
            return None

        return numeric_value

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, torch.Tensor):
        tensor_value = (
            value
            .detach()
            .cpu()
        )

        if tensor_value.ndim == 0:
            return make_json_serializable(
                tensor_value.item()
            )

        return make_json_serializable(
            tensor_value.tolist()
        )

    if isinstance(value, Path):
        return str(value)

    return str(value)


metrics_output = {
    "trainer_metrics": test_metrics,
    "detailed_metrics": (
        test_results.get("metrics", {})
        if isinstance(test_results, dict)
        else {}
    ),
    "per_label": (
        test_results.get("per_label", [])
        if isinstance(test_results, dict)
        else []
    ),
}

with (
    final_dir / "test_metrics.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        make_json_serializable(
            metrics_output
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "\nModèle sauvegardé dans :",
    final_dir,
)

Variables de test disponibles.

ÉVALUATION DU MODÈLE MULTILABEL


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Subset Accuracy,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Weighted F1,Average Precision Micro,Average Precision Macro,Empty Prediction Rate
No log,1.227960,0,0.000000,0.110934,0.518657,0.182774,0.114257,0.566070,0.111910,0.156151,0.110970,0.143018,0.000000


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...


Test multilabel
Seuil                 : 0.500
Nombre d'exemples     : 117
Nombre de compétences : 20

Métriques globales :
Précision micro : 0.1109
Rappel micro    : 0.5187
F1 micro        : 0.1828

Précision macro : 0.1143
Rappel macro    : 0.5661
F1 macro        : 0.1119

F1 samples      : 0.1751
Exact match     : 0.0000

Nombre moyen de compétences réelles par formation : 2.29
Nombre moyen de compétences prédites par formation : 10.71

Compétences les plus difficiles :


,competence,support,precision,recall,f1,predictions_positives
0,Reinforcement Learning,1,0.000000,0.000000,0.000000,1
1,NLP / Traitement du langage,2,0.000000,0.000000,0.000000,0
2,MLOps / Deploiement,6,0.000000,0.000000,0.000000,0
3,Deep Learning,8,0.000000,0.000000,0.000000,0
4,SQL / Data Engineering,14,0.000000,0.000000,0.000000,0
5,Big Data,15,0.000000,0.000000,0.000000,1
6,Automatisation,18,0.000000,0.000000,0.000000,0
7,Computer Vision,2,0.017241,1.000000,0.033898,116
8,IA Generative,51,1.000000,0.019608,0.038462,1
9,No-code / Low-code,4,0.034188,1.000000,0.066116,117



Rapport de classification :
                             precision    recall  f1-score   support

             Automatisation       0.00      0.00      0.00        18
                   Big Data       0.00      0.00      0.00        15
            Computer Vision       0.02      1.00      0.03         2
           Data Engineering       0.08      1.00      0.14         9
               Data Science       0.20      1.00      0.33        23
              Deep Learning       0.00      0.00      0.00         8
          Ethique IA & RGPD       0.26      1.00      0.42        31
       Gestion de projet IA       0.13      0.48      0.21        23
              IA Generative       1.00      0.02      0.04        51
     LangChain / Agents RAG       0.05      1.00      0.10         6
           Machine Learning       0.15      1.00      0.26        17
        MLOps / Deploiement       0.00      0.00      0.00         6
NLP / Traitement du langage       0.00      0.00      0.00         2
    

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modèle sauvegardé dans : /teamspace/studios/this_studio/deepforma/models/ia-classifier-v2/final
